In [25]:
import cv2
from ultralytics import YOLO
import easyocr
import  os, urllib.request

In [26]:
# Load models
vehicle_model = YOLO("yolov8n.pt")
if not os.path.exists("yolov8n-license-plate.pt"):
    urllib.request.urlretrieve(
        "https://huggingface.co/Koushim/yolov8-license-plate-detection/resolve/main/best.pt",
        "yolov8n-license-plate.pt"
    )
plate_model = YOLO("yolov8n-license-plate.pt")

In [27]:

reader = easyocr.Reader(['en'])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [28]:
# Đóng cap cũ nếu tồn tại (tránh lỗi Jupyter)
if 'cap' in globals():
    cap.release()

cap = cv2.VideoCapture("video.mp4")

frame_id = 0
vehicle_plate_cache = {}  # {vehicle_box: plate_text}


In [29]:
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)

In [30]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1

    vehicle_results = vehicle_model(frame, imgsz=416, conf=0.4)
    plate_results = plate_model(frame, imgsz=416, conf=0.4)

    vehicles = []
    plates = []

    # Collect vehicles
    for r in vehicle_results:
        for box in r.boxes:
            cls = int(box.cls[0])
            if cls in [2, 3, 5, 7]:
                vehicles.append(tuple(map(int, box.xyxy[0])))

    # Collect plates
    for r in plate_results:
        for box in r.boxes:
            plates.append(tuple(map(int, box.xyxy[0])))

    # Assign plate to vehicle
    for vbox in vehicles:
        assigned_plate = None
        max_iou = 0

        for pbox in plates:
            val = iou(vbox, pbox)
            if val > max_iou:
                max_iou = val
                assigned_plate = pbox

        plate_text = vehicle_plate_cache.get(vbox, "")

        # OCR mỗi 6 frame
        if assigned_plate and frame_id % 6 == 0:
            x1, y1, x2, y2 = assigned_plate
            plate_roi = frame[y1:y2, x1:x2]
            if plate_roi.size > 0:
                ocr = reader.readtext(plate_roi)
                if ocr:
                    plate_text = ocr[0][1]
                    vehicle_plate_cache[vbox] = plate_text

        # Draw vehicle
        cv2.rectangle(frame, (vbox[0], vbox[1]), (vbox[2], vbox[3]), (0,255,0), 2)

        if plate_text:
            cv2.putText(
                frame,
                plate_text,
                (vbox[0], vbox[1]-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0,0,255),
                2
            )

    cv2.imshow("Vehicle License Plate System", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break


0: 256x416 9 cars, 1 motorcycle, 1 bus, 1 truck, 27.5ms
Speed: 8.3ms preprocess, 27.5ms inference, 0.6ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 (no detections), 23.2ms
Speed: 1.1ms preprocess, 23.2ms inference, 0.3ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 12 cars, 1 motorcycle, 1 bus, 1 truck, 98.3ms
Speed: 2.7ms preprocess, 98.3ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 (no detections), 93.9ms
Speed: 2.0ms preprocess, 93.9ms inference, 1.1ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 10 cars, 1 motorcycle, 1 truck, 102.8ms
Speed: 2.9ms preprocess, 102.8ms inference, 2.0ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 (no detections), 130.1ms
Speed: 2.5ms preprocess, 130.1ms inference, 1.2ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 1 person, 11 cars, 1 motorcycle, 1 bus, 2 trucks, 193.0ms
Speed: 7.8ms preprocess, 193.0ms inference, 1.6ms postprocess per imag

c:\Users\LAN ANH\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


0: 256x416 11 cars, 1 motorcycle, 1 bus, 1 truck, 27.5ms
Speed: 1.2ms preprocess, 27.5ms inference, 0.8ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 2 license_plates, 24.4ms
Speed: 1.6ms preprocess, 24.4ms inference, 0.5ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 1 person, 11 cars, 1 bus, 1 truck, 26.0ms
Speed: 1.7ms preprocess, 26.0ms inference, 0.6ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 1 license_plate, 21.0ms
Speed: 1.1ms preprocess, 21.0ms inference, 0.6ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 10 cars, 2 trucks, 29.4ms
Speed: 1.2ms preprocess, 29.4ms inference, 0.5ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 1 license_plate, 25.1ms
Speed: 1.2ms preprocess, 25.1ms inference, 0.6ms postprocess per image at shape (1, 3, 256, 416)

0: 256x416 1 person, 9 cars, 1 bus, 2 trucks, 29.0ms
Speed: 1.2ms preprocess, 29.0ms inference, 0.6ms postprocess per image at shape (1, 3, 256, 416)

0: 256x4

In [32]:
cap.release()
cv2.destroyAllWindows()